[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/16_hyperparameter_optimization/02_successive_halving_hyperband.ipynb)

# Bài 02 — Successive Halving và Hyperband

**🎯 Mục tiêu**
- Ý tưởng **multi-fidelity**: đánh giá rẻ (ít dữ liệu / ít vòng lặp) để loại sớm cấu hình kém.
- Cài đặt **Successive Halving** from scratch; dùng `HalvingRandomSearchCV`.
- **Hyperband**: chạy nhiều "bracket" để cân bằng giữa số cấu hình và ngân sách mỗi cấu hình.
- So sánh thời gian/chất lượng với random search; rủi ro khi đánh giá rẻ không phản ánh đánh giá đầy đủ.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/16_hyperparameter_optimization

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.experimental import enable_halving_search_cv  # noqa: F401  (bật API thử nghiệm)
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import HalvingRandomSearchCV, ParameterSampler, RandomizedSearchCV, StratifiedKFold, train_test_split

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
churn = make_customer_churn(n=9000, seed=9)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float).to_numpy()
y = churn["churn"].to_numpy()
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
space = {"learning_rate": loguniform(0.005, 0.5), "max_leaf_nodes": randint(2, 128), "min_samples_leaf": randint(2, 300),
         "l2_regularization": loguniform(1e-4, 10)}

## 1. Ý tưởng: đánh giá rẻ trước, loại sớm

Phần lớn cấu hình ngẫu nhiên là **tệ** — và thường đã lộ ra là tệ khi train với **một phần nhỏ dữ liệu** (hoặc ít vòng lặp). Thay vì cho mọi cấu hình ngân sách đầy đủ:

**Successive Halving** (Jamieson & Talwalkar, 2016) với hệ số $\eta$ (thường 3):
1. Lấy $n$ cấu hình ngẫu nhiên, mỗi cấu hình ngân sách nhỏ $r$.
2. Giữ lại $1/\eta$ cấu hình tốt nhất, nhân ngân sách lên $\eta$ lần.
3. Lặp tới khi còn 1 cấu hình (hoặc hết ngân sách tối đa).

Tổng chi phí mỗi vòng xấp xỉ bằng nhau (ít cấu hình × nhiều ngân sách).

In [ ]:
def evaluate(params, n_samples, seed=0):
    """Train trên n_samples mẫu đầu (đã xáo trộn) của tập train, đánh giá ROC-AUC trên validation."""
    idx = np.random.default_rng(seed).permutation(len(X_tr))[:n_samples]
    m = HistGradientBoostingClassifier(max_iter=150, early_stopping=False, random_state=0, **params).fit(X_tr[idx], y_tr[idx])
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])


def successive_halving(n_configs=27, min_samples=300, eta=3, seed=0):
    configs = list(ParameterSampler(space, n_iter=n_configs, random_state=seed))
    ids = list(range(len(configs)))
    budget, history, cost = min_samples, [], 0
    while True:
        scores = [evaluate(c, min(budget, len(X_tr))) for c in configs]
        cost += len(configs) * min(budget, len(X_tr))
        history.append((min(budget, len(X_tr)), len(configs), max(scores), dict(zip(ids, scores))))
        if len(configs) == 1 or budget >= len(X_tr):
            return configs[int(np.argmax(scores))], history, cost
        keep = np.argsort(scores)[::-1][: max(1, len(configs) // eta)]
        configs, ids = [configs[i] for i in keep], [ids[i] for i in keep]
        budget *= eta


start = time.perf_counter()
best_sh, hist, cost_sh = successive_halving()
t_sh = time.perf_counter() - start
for budget, n_cfg, best, _ in hist:
    print(f"ngân sách {budget:5d} mẫu: {n_cfg:2d} cấu hình, tốt nhất = {best:.4f}")
print(f"SH: {t_sh:.1f}s | tổng 'mẫu-train' = {cost_sh:,}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for cid in range(27):
    pts = [(b, scores_by_id[cid]) for b, _, _, scores_by_id in hist if cid in scores_by_id]
    survived = len(pts) == len(hist)
    ax.plot(*zip(*pts), "o-", ms=4, lw=1.8 if survived else 0.8, color="C2" if survived else "0.6", alpha=1 if survived else 0.6)
ax.set_xscale("log")
ax.set_xlabel("ngân sách: số mẫu train (thang log)")
ax.set_ylabel("ROC-AUC validation")
ax.set_title("Successive halving: mỗi đường là một cấu hình, bị loại thì dừng (xanh = cấu hình được chọn)")
plt.show()

In [ ]:
start = time.perf_counter()
configs = list(ParameterSampler(space, n_iter=27, random_state=0))
full_scores = [evaluate(c, len(X_tr)) for c in configs]
t_full = time.perf_counter() - start
best_full = configs[int(np.argmax(full_scores))]
print(f"Random search đầy đủ (27 cấu hình × {len(X_tr)} mẫu): {t_full:.1f}s | tổng 'mẫu-train' = {27 * len(X_tr):,}")
print(f"ROC-AUC (đánh giá đầy đủ): SH chọn = {evaluate(best_sh, len(X_tr)):.4f} | random đầy đủ chọn = {max(full_scores):.4f}")
rank_of_sh = int(np.sum(np.array(full_scores) > evaluate(best_sh, len(X_tr)))) + 1
print(f"cấu hình SH chọn xếp hạng {rank_of_sh}/27 nếu đánh giá đầy đủ")

Successive halving tìm được cấu hình tốt (thường nằm trong top) với **một phần nhỏ** chi phí. Rủi ro: cấu hình cần nhiều dữ liệu mới tỏa sáng (vd cây rất lớn + regularization mạnh) có thể bị loại oan ở vòng đầu — vì vậy ngân sách nhỏ nhất không được **quá** nhỏ.

## 2. `HalvingRandomSearchCV` trong scikit-learn

In [ ]:
cv = StratifiedKFold(3, shuffle=True, random_state=0)
base = HistGradientBoostingClassifier(max_iter=150, early_stopping=False, random_state=0)
start = time.perf_counter()
halving = HalvingRandomSearchCV(base, space, n_candidates=54, factor=3, resource="n_samples", min_resources=250,
                                cv=cv, scoring="roc_auc", random_state=0).fit(X_tr, y_tr)
t_halving = time.perf_counter() - start
start = time.perf_counter()
rand = RandomizedSearchCV(base, space, n_iter=54, cv=cv, scoring="roc_auc", random_state=0).fit(X_tr, y_tr)
t_rand = time.perf_counter() - start
for name, s, t in [("HalvingRandomSearchCV", halving, t_halving), ("RandomizedSearchCV", rand, t_rand)]:
    print(f"{name:22s} {t:5.1f}s | ROC-AUC validation của cấu hình chọn = {roc_auc_score(y_val, s.predict_proba(X_val)[:, 1]):.4f}")
pd.DataFrame(halving.cv_results_).groupby("iter").agg(n_cấu_hình=("params", "size"), n_resources=("n_resources", "first"),
                                                       best=("mean_test_score", "max")).round(4)

Halving nhanh hơn hơn 2 lần và ở đây chọn được cấu hình tốt ngang random search đầy đủ — không phải lúc nào cũng vậy: cấu hình cần nhiều dữ liệu mới tỏa sáng có thể bị loại oan ở vòng đầu (đánh đổi giữa chi phí và độ chắc chắn). Lợi ích tăng mạnh khi mỗi lần train rất đắt (dữ liệu lớn, mạng neural).

Resource không nhất thiết là số mẫu: với boosting/mạng neural có thể là **số vòng lặp/epoch** (`resource="max_iter"`) — đánh giá sau ít epoch rồi loại sớm.

## 3. Hyperband

Successive halving có một tham số khó chọn: bắt đầu với **nhiều cấu hình × ngân sách nhỏ** (thăm dò rộng nhưng dễ loại oan) hay **ít cấu hình × ngân sách lớn** (an toàn nhưng thăm dò hẹp)? **Hyperband** (Li et al., 2018) chạy **nhiều bracket** với các mức đánh đổi khác nhau và lấy kết quả tốt nhất:

In [ ]:
def hyperband_plan(R=8100, eta=3, r_min=100):
    s_max = int(np.floor(np.log(R / r_min) / np.log(eta)))
    rows = []
    for s in range(s_max, -1, -1):
        n = int(np.ceil((s_max + 1) / (s + 1) * eta**s))
        r = R * eta ** (-s)
        rows.append({"bracket s": s, "số cấu hình ban đầu": n, "ngân sách ban đầu": int(r),
                     "các vòng (cấu hình × ngân sách)": " → ".join(f"{int(n * eta ** -i)}×{int(r * eta**i)}" for i in range(s + 1))})
    return pd.DataFrame(rows)


hyperband_plan()

Bracket đầu: rất nhiều cấu hình, ngân sách nhỏ (thăm dò mạnh); bracket cuối: ít cấu hình, ngân sách đầy đủ (= random search). Optuna triển khai ý tưởng này qua `HyperbandPruner` (bài 03).

## 🧠 Kiểm tra nhanh

1. Successive halving với 81 cấu hình, $\eta = 3$: qua mỗi vòng còn lại bao nhiêu cấu hình?
2. "Resource" trong successive halving có thể là gì?
3. Khi nào đánh giá rẻ (ít dữ liệu) **không** phản ánh đúng thứ hạng cấu hình?
4. Hyperband giải quyết vấn đề gì của successive halving?

## 📌 Tóm tắt

- Multi-fidelity: đánh giá rẻ để loại sớm → tiết kiệm phần lớn chi phí.
- Successive halving: giữ $1/\eta$ tốt nhất, nhân ngân sách $\eta$; Hyperband: nhiều bracket để không phải chọn đánh đổi.
- Rủi ro: thứ hạng ở ngân sách nhỏ khác ngân sách lớn → đừng để ngân sách nhỏ nhất quá nhỏ.

## 📚 Tài liệu tham khảo

- Jamieson & Talwalkar (2016) — *Non-stochastic Best Arm Identification and Hyperparameter Optimization*
- Li et al. (2018) — *Hyperband: A Novel Bandit-Based Approach to Hyperparameter Optimization*
- scikit-learn User Guide — Searching for optimal parameters with successive halving